In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Import the dataset


In [ ]:
df = pd.read_csv("../datasets/spotify_tracks.csv")
df.head()

,track_id,track_name,artist_name,album_name,release_year,genre,popularity,duration_ms,explicit,danceability,...,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,time_signature
0,P3fAbnFbmOHnKYaXRvj7uf,One Dance (Acoustic Version),Alex Rodriguez,The Night Album,2024,metal,14,189042,True,0.427723,...,-4.702460,0.050635,0.239506,0.181395,0.133053,0.431384,141.048735,6,0,4
1,M2wleOV911xCZkwPRQeNHp,Forever Song (Remix),Desert Wind,Burning Soul,2019,rock,11,186805,True,0.448634,...,-7.110031,0.000000,0.044463,0.097818,0.435949,0.559135,131.833287,0,1,5
2,4JSnE2NiiUHUAKw9iEU1jj,Last Mountain,The Midnight,The Night Album,2022,k-pop,23,121814,False,0.707923,...,-7.305120,0.144091,0.118380,0.000000,0.262254,0.516873,127.132954,2,1,5
3,2UVvsjaSS8VFgM0Fmxk754,Falling Star (Live),Phantom Keys,Phantom's Greatest Hits,2024,latin,34,216049,False,0.846237,...,-9.527256,0.006668,0.272844,0.000000,0.045332,0.667911,93.041715,0,1,6
4,EeVVhDIq2AnHTmt9OBGhnu,Rising Moon (feat. someone),Desert Wind,Rising Soul,2010,latin,31,156170,False,0.943720,...,-9.017653,0.057632,0.219752,0.098044,0.132083,0.772151,93.404975,7,1,4


### We want to check some of the properties of the dataset

In [8]:
# number of rows/cols
len(df)
len(df.columns)

21

In [15]:
df['genre'].unique() # number of unique genres
df.columns # list of features

Index(['track_id', 'track_name', 'artist_name', 'album_name', 'release_year',
       'genre', 'popularity', 'duration_ms', 'explicit', 'danceability',
       'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'key', 'mode', 'time_signature'],
      dtype='object')

In [40]:
X = df.drop(columns=['track_id', 'track_name', 'artist_name', 'album_name', 'genre']) # features to use

# target column
y = df["genre"] 

# we need to turn classes into numeric values
le = LabelEncoder()
y = le.fit_transform(y)


### Begin by prepping the model

In [38]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# vectorize text columns
track_vec = CountVectorizer(max_features=1000) # adjust feature size
album_vec = CountVectorizer(max_features=1000)

# these are now large sparse matrices
X_track = track_vec.fit_transform(df['track_name'].fillna(''))
X_album = album_vec.fit_transform(df['album_name'].fillna(''))

# numeric features
numeric_cols = [
    'release_year', 'popularity', 'duration_ms', 'explicit',
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'key', 'mode', 'time_signature'
]

df['explicit'] = df['explicit'].map({True:1, False:0}).fillna(0) # convert boolean to numeric and fill NaNs
X_numeric = df[numeric_cols].fillna(0).astype(float).values

X = hstack([X_numeric, X_track, X_album]) # x
y = df['genre']
le = LabelEncoder()
y = le.fit_transform(y)

# train / test split
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% trainval, 20% test
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42) #60% train, 20% val, 20% test


In [42]:
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
best_k = 0
best_acc = 0

for k in range(1, 100, 20):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_val, knn.predict(X_val))
    
    
    if acc > best_acc:
        best_acc = acc
        best_k = k

print("Best k:", best_k)
print("Best accuracy:", best_acc)
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

y_pred_names = le.inverse_transform(y_pred)
y_test_names = le.inverse_transform(y_test)
print(classification_report(y_test_names, y_pred_names))

Best k: 81
Best accuracy: 0.1263
               precision    recall  f1-score   support

      ambient       0.00      0.00      0.00       275
        blues       0.00      0.00      0.00       150
    classical       0.03      0.00      0.00       479
      country       0.06      0.01      0.01       532
drum-and-bass       0.00      0.00      0.00       184
   electronic       0.07      0.03      0.04       737
         folk       0.00      0.00      0.00       358
       gospel       0.00      0.00      0.00       193
      hip-hop       0.13      0.31      0.18      1169
        indie       0.00      0.00      0.00       365
         jazz       0.00      0.00      0.00       495
        k-pop       0.07      0.01      0.01       436
        latin       0.00      0.00      0.00       362
        metal       0.07      0.01      0.01       443
          pop       0.14      0.50      0.22      1360
         punk       0.00      0.00      0.00       200
          r&b       0.08      0

c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\zohai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo